# MLflow Models: Model Deployment and Serving

In this notebook we will cover:
1. Local model serving
2. REST API testing
3. Batch inference
4. Performance testing
5. Production deployment patterns

## Deployment Options

MLflow models can be deployed in multiple ways:
- **Local REST API** - `mlflow models serve`
- **Batch inference** - Load model and process files
- **Docker containers** - Containerized deployment
- **Cloud platforms** - AWS SageMaker, Azure ML, etc.
- **Kubernetes** - Scalable production deployment

## 1. Import Libraries

In [ ]:
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import requests
import json
import time
import sys

sys.path.append('../src')
from data_loader import load_sample_data, get_sample_input
from model_inspector import inspect_model_structure, export_model
from model_client import MLflowModelClient

print(f"MLflow version: {mlflow.__version__}")

## 2. Prepare and Train Model

In [ ]:
X_train, X_test, y_train, y_test, feature_names, target_names = load_sample_data('wine')

print(f"Dataset: Wine classification")
print(f"Features: {len(feature_names)}")
print(f"Classes: {len(target_names)}")
print(f"Training samples: {len(X_train)}")

In [ ]:
mlflow.set_tracking_uri("file:./mlruns")
mlflow.set_experiment("05_Model_Deployment")

with mlflow.start_run(run_name="production_model") as run:
    model = RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        min_samples_split=5,
        random_state=42
    )
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    
    X_sample = pd.DataFrame(X_train[:5], columns=feature_names)
    y_sample_pred = model.predict(X_sample)
    signature = infer_signature(X_sample, y_sample_pred)
    
    input_example = X_sample.iloc[:1]
    
    mlflow.log_params({
        "n_estimators": 100,
        "max_depth": 10,
        "min_samples_split": 5
    })
    mlflow.log_metric("accuracy", accuracy)
    
    mlflow.sklearn.log_model(
        sk_model=model,
        artifact_path="model",
        signature=signature,
        input_example=input_example
    )
    
    run_id = run.info.run_id
    model_uri = f"runs:/{run_id}/model"
    
    print(f"Run ID: {run_id}")
    print(f"Model URI: {model_uri}")
    print(f"Accuracy: {accuracy:.4f}")

## 3. Export Model for Production

In [ ]:
exported_path = export_model(
    model_uri=model_uri,
    output_dir="../models",
    model_name="wine_classifier_prod"
)

print(f"Model exported to: {exported_path}")
print(f"\nModel structure:")
inspect_model_structure(exported_path, verbose=True)

## 4. Local Model Serving

### Start Model Server

To start the model server, run this command in a **separate terminal**:

```bash
mlflow models serve -m runs:/{run_id}/model -p 5000
```

Or using the exported model:

```bash
mlflow models serve -m ../models/wine_classifier_prod -p 5000
```

The server will start at `http://localhost:5000`

In [ ]:
print("To start the model server, run this command in a separate terminal:\n")
print(f"mlflow models serve -m runs:/{run_id}/model -p 5000")
print("\nOr using exported model:")
print(f"mlflow models serve -m {exported_path} -p 5000")

## 5. Testing REST API

**Note:** The following cells require the model server to be running!

In [ ]:
server_url = "http://localhost:5000"

try:
    response = requests.get(f"{server_url}/ping", timeout=2)
    if response.status_code == 200:
        print(" Server is running!")
    else:
        print(f" Server responded with status {response.status_code}")
except requests.exceptions.ConnectionError:
    print(" Server is not running. Please start the server first.")
except Exception as e:
    print(f" Error: {e}")

### Single Prediction

In [ ]:
X_test_df = pd.DataFrame(X_test, columns=feature_names)
test_data = X_test_df.iloc[:1].to_dict(orient='split')

payload = {
    "dataframe_split": test_data
}

print("Request payload:")
print(json.dumps(payload, indent=2)[:500] + "...")

try:
    response = requests.post(
        f"{server_url}/invocations",
        headers={"Content-Type": "application/json"},
        data=json.dumps(payload),
        timeout=5
    )
    
    if response.status_code == 200:
        prediction = response.json()
        print(f"\n Prediction: {prediction}")
        print(f"Actual label: {y_test[0]}")
    else:
        print(f" Error: {response.status_code}")
        print(response.text)
except Exception as e:
    print(f" Cannot connect to server: {e}")

### Batch Prediction

In [ ]:
batch_size = 10
batch_data = X_test_df.iloc[:batch_size].to_dict(orient='split')

payload = {
    "dataframe_split": batch_data
}

try:
    start_time = time.time()
    response = requests.post(
        f"{server_url}/invocations",
        headers={"Content-Type": "application/json"},
        data=json.dumps(payload),
        timeout=10
    )
    elapsed_time = time.time() - start_time
    
    if response.status_code == 200:
        predictions = response.json()
        print(f" Batch predictions ({batch_size} samples):")
        print(f"Predictions: {predictions}")
        print(f"Actual labels: {y_test[:batch_size].tolist()}")
        print(f"\nTime: {elapsed_time:.4f} seconds")
        print(f"Average time per sample: {elapsed_time/batch_size*1000:.2f} ms")
    else:
        print(f" Error: {response.status_code}")
except Exception as e:
    print(f" Cannot connect to server: {e}")

## 6. Using Model Client Utility

In [ ]:
try:
    client = MLflowModelClient(server_url)
    
    if client.ping():
        print(" Server is healthy\n")
        
        single_sample = X_test_df.iloc[:1]
        prediction = client.predict_single(single_sample, feature_names)
        print(f"Single prediction: {prediction}")
        print(f"Actual: {y_test[0]}")
        
        batch_samples = X_test_df.iloc[:5]
        batch_predictions = client.predict(batch_samples)
        print(f"\nBatch predictions: {batch_predictions}")
        print(f"Actual: {y_test[:5].tolist()}")
    else:
        print(" Server is not responding")
except Exception as e:
    print(f" Error: {e}")

## 7. Batch Inference (Without Server)

In [ ]:
loaded_model = mlflow.pyfunc.load_model(model_uri)

start_time = time.time()
predictions = loaded_model.predict(X_test_df)
elapsed_time = time.time() - start_time

accuracy = (predictions == y_test).mean()

print(f"Batch inference results:")
print(f"Samples processed: {len(X_test)}")
print(f"Time: {elapsed_time:.4f} seconds")
print(f"Throughput: {len(X_test)/elapsed_time:.0f} samples/second")
print(f"Accuracy: {accuracy:.4f}")

## 8. Performance Comparison

In [ ]:
def benchmark_batch_inference(model, data, n_runs=5):
    times = []
    for _ in range(n_runs):
        start = time.time()
        _ = model.predict(data)
        times.append(time.time() - start)
    return np.mean(times), np.std(times)

batch_sizes = [1, 10, 50, 100]
results = []

print("Performance benchmark:")
print("=" * 60)

for batch_size in batch_sizes:
    batch_data = X_test_df.iloc[:batch_size]
    mean_time, std_time = benchmark_batch_inference(loaded_model, batch_data, n_runs=10)
    
    throughput = batch_size / mean_time
    latency_per_sample = mean_time / batch_size * 1000  # ms
    
    results.append({
        'batch_size': batch_size,
        'mean_time': mean_time,
        'std_time': std_time,
        'throughput': throughput,
        'latency_ms': latency_per_sample
    })
    
    print(f"Batch size: {batch_size:3d} | "
          f"Time: {mean_time:.4f}±{std_time:.4f}s | "
          f"Throughput: {throughput:6.0f} samples/s | "
          f"Latency: {latency_per_sample:5.2f} ms/sample")

results_df = pd.DataFrame(results)
print("\n" + "=" * 60)
print(results_df.to_string(index=False))

## 9. Production Deployment Checklist

In [ ]:
deployment_checklist = {
    "Model Validation": [
        " Model accuracy verified on test set",
        " Model signature defined",
        " Input example provided",
        " Model structure inspected"
    ],
    "Performance": [
        " Batch inference benchmarked",
        " Latency requirements met",
        " Throughput requirements met",
        " Load testing performed (recommended)"
    ],
    "Deployment": [
        " Model exported to models/ directory",
        " REST API tested locally",
        " Docker image created (optional)",
        " Monitoring configured (recommended)"
    ],
    "Documentation": [
        " Model parameters logged",
        " Metrics logged",
        " Input/output schema documented",
        " API documentation created (recommended)"
    ]
}

print("Production Deployment Checklist")
print("=" * 60)
for category, items in deployment_checklist.items():
    print(f"\n{category}:")
    for item in items:
        print(f"  {item}")

## 10. Next Steps for Production

### Docker Deployment

```bash
# Build Docker image
mlflow models build-docker -m runs:/{run_id}/model -n wine-classifier

# Run container
docker run -p 5000:8080 wine-classifier
```

### Cloud Deployment Examples

**AWS SageMaker:**
```python
import mlflow.sagemaker
mlflow.sagemaker.deploy(
    model_uri=model_uri,
    app_name="wine-classifier",
    region_name="us-east-1"
)
```

**Azure ML:**
```python
import mlflow.azureml
mlflow.azureml.deploy(
    model_uri=model_uri,
    workspace=workspace,
    deployment_name="wine-classifier"
)
```

In [ ]:
print("Deployment Commands Summary:")
print("=" * 60)
print("\n1. Local serving:")
print(f"   mlflow models serve -m {exported_path} -p 5000")
print("\n2. Build Docker image:")
print(f"   mlflow models build-docker -m {model_uri} -n wine-classifier")
print("\n3. Test API:")
print("   curl http://localhost:5000/ping")
print("\n4. Batch inference:")
print("   python batch_inference.py --model-uri " + model_uri)

## Conclusions

In this notebook we learned:

1.  Local model serving with MLflow
2.  REST API testing (single and batch)
3.  Batch inference without server
4.  Performance benchmarking
5.  Production deployment checklist
6.  Docker and cloud deployment options

### Deployment Patterns:
- **Real-time API** - Low latency, single predictions
- **Batch inference** - High throughput, process files
- **Streaming** - Process data streams
- **Edge deployment** - Mobile/IoT devices

### Production Considerations:
- Monitoring and logging
- Model versioning
- A/B testing
- Rollback strategies
- Scalability and load balancing

## Congratulations! 🎉

You have completed all 5 notebooks on MLflow Models:
1.  Basic Model Logging
2.  Model Signatures
3.  Model Flavors
4.  Custom PyFunc Models
5.  Model Deployment

You now have a solid understanding of MLflow Models and are ready for production ML deployments!